# CIPHER — Kaggle Benchmark Notebook
**Calibrated Introspection via Partially Hidden Environment Rules**

This notebook runs the CIPHER benchmark against as many frontier LLMs as possible
and registers the results on Kaggle Benchmarks.

**Dataset expected at** `/kaggle/input/cipher-benchmark/`
- `cipher/` — the Python package (world.py, generator.py, simulator.py, scorer.py, schema.py, prompt.py, flavor.py, optimal.py)
- `scripts/evaluate.py`
- `data/instances.jsonl` — 1 000 pre-generated instances (seed=2026)
- `README.md`

**Scoring dimensions** (all normalized to [0, 1]):
| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Objective | 35 % | Final plan quality vs oracle |
| Calibration | 25 % | Brier score on self-knowledge claims |
| Attention | 20 % | Rank correlation on unknown importance |
| Executive | 20 % | Plan structure, risks named, alternatives given |

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

pip("kaggle-benchmarks")
pip("google-genai")          # google.genai (new unified SDK)
pip("anthropic")              # Anthropic / Claude
pip("openai")                 # OpenAI / GPT-4o
pip("huggingface_hub")        # HF Inference Providers
pip("tqdm", "pandas", "matplotlib", "tabulate")
print("All packages installed.")

In [ ]:
# ── 2. Path setup & core imports ─────────────────────────────────────────────
import os, sys, json, random, time, datetime
from typing import Any, Dict, List, Optional

# Locate the cipher package.  Try dataset path first, then local repo.
DATASET_ROOT = "/kaggle/input/cipher-benchmark"
LOCAL_ROOT   = os.path.abspath(os.path.join(os.path.dirname("__file__"), "."))

for root in [DATASET_ROOT, LOCAL_ROOT]:
    if os.path.isdir(os.path.join(root, "cipher")):
        sys.path.insert(0, root)
        print(f"cipher package found at: {root}")
        break
else:
    raise RuntimeError(
        "Cannot find cipher/ package. "
        "Attach the cipher-benchmark dataset to this notebook."
    )

from cipher import build_prompt
from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect, Action
from cipher.simulator import run_plan
from cipher.schema import validate_response
from cipher.scorer import score_response
from cipher.optimal import oracle_score

print("cipher imports OK.")

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────────────────

# Path to the pre-generated dataset
DATA_PATH = os.path.join(DATASET_ROOT, "data", "instances.jsonl")
if not os.path.exists(DATA_PATH):
    # Fallback to local
    DATA_PATH = os.path.join(LOCAL_ROOT, "data", "instances.jsonl")

# How many instances to evaluate per model.
# 50 is a good balance of statistical signal vs. cost / time.
# Set to None to run all 1 000.
EVAL_LIMIT = 50

# API keys — Kaggle secrets or environment variables
# In a Kaggle notebook, add secrets under Add-ons → Secrets
def _get_secret(name: str) -> str:
    """Return env-var or Kaggle secret, or '' if not found."""
    if val := os.environ.get(name, ""):
        return val
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

MODEL_PROXY_API_KEY = _get_secret("MODEL_PROXY_API_KEY")  # Kaggle-provided quota
ANTHROPIC_API_KEY   = _get_secret("ANTHROPIC_API_KEY")
OPENAI_API_KEY      = _get_secret("OPENAI_API_KEY")
HF_TOKEN            = _get_secret("HF_TOKEN")

print(f"MODEL_PROXY_API_KEY : {'set' if MODEL_PROXY_API_KEY else 'MISSING'}")
print(f"ANTHROPIC_API_KEY   : {'set' if ANTHROPIC_API_KEY   else 'MISSING'}")
print(f"OPENAI_API_KEY      : {'set' if OPENAI_API_KEY      else 'MISSING'}")
print(f"HF_TOKEN            : {'set' if HF_TOKEN            else 'MISSING'}")
print(f"\nDATA_PATH : {DATA_PATH}  (exists={os.path.exists(DATA_PATH)})")
print(f"EVAL_LIMIT: {EVAL_LIMIT}")

In [ ]:
# ── 4. Load instances ────────────────────────────────────────────────────────

def load_instances(path: str, limit: Optional[int] = None) -> List[Dict]:
    with open(path) as f:
        recs = [json.loads(line) for line in f if line.strip()]
    if limit:
        recs = recs[:limit]
    return recs

def _instance_from_record(rec: Dict[str, Any]) -> Instance:
    """Re-hydrate a serialised JSONL record into a live Instance object."""
    hidden = rec["hidden"]
    hidden_rules_lookup = {
        h["rule_name"]: set(h["hidden"])
        for h in hidden.get("hidden_fields", [])
    }
    rules = []
    for r in hidden["rules"]:
        hides  = hidden_rules_lookup.get(r["name"], set())
        t_raw  = r["trigger"]
        e_raw  = r["effect"]
        trigger = Trigger(
            kind=t_raw["kind"], i=t_raw["i"], j=t_raw.get("j", -1),
            k=t_raw.get("k", 0),
            hidden_kind="trigger_kind" in hides,
            hidden_k="trigger_k"   in hides,
        )
        effect = Effect(
            kind=e_raw["kind"], target=e_raw["target"],
            delta=e_raw.get("delta", 0), source=e_raw.get("source", -1),
            hidden_kind="effect_kind"  in hides,
            hidden_delta="effect_delta" in hides,
        )
        rules.append(Rule(name=r["name"], trigger=trigger, effect=effect))

    initial = State(tuple(
        EntityState(phase=e["phase"], flux=e["flux"])
        for e in hidden["initial_state"]
    ))
    world = World(initial=initial, rules=tuple(rules), horizon=hidden["horizon"])
    return Instance(
        id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
        world=world, public_rule_descriptions=[],
        hidden_fields=hidden.get("hidden_fields", []),
        metacog_ground_truth=hidden["metacog_ground_truth"],
        true_unknown_ranking=hidden["true_unknown_ranking"],
        oracle_objective=hidden.get("oracle_best"),
    )


ALL_RECORDS  = load_instances(DATA_PATH)
EVAL_RECORDS = load_instances(DATA_PATH, limit=EVAL_LIMIT)
print(f"Loaded {len(ALL_RECORDS)} total  |  evaluating {len(EVAL_RECORDS)} per model")

In [ ]:
# ── 5. Shared scoring helper ─────────────────────────────────────────────────

def score_text_response(raw_text: str, rec: Dict[str, Any]) -> Dict[str, Any]:
    """Parse JSON out of a model reply and return the score breakdown dict."""
    start, end = raw_text.find("{"), raw_text.rfind("}")
    if start == -1 or end == -1:
        return {"error": "no_json", "composite": 0.0,
                "objective": 0.0, "calibration": 0.0,
                "attention": 0.0, "executive": 0.0}
    try:
        raw = json.loads(raw_text[start:end + 1])
    except json.JSONDecodeError as exc:
        return {"error": str(exc), "composite": 0.0,
                "objective": 0.0, "calibration": 0.0,
                "attention": 0.0, "executive": 0.0}
    inst     = _instance_from_record(rec)
    resp     = validate_response(raw)
    best_obj = rec["hidden"].get("oracle_best")
    worst_obj= rec["hidden"].get("oracle_worst")
    bd       = score_response(resp, inst, best_obj=best_obj, worst_obj=worst_obj)
    return bd.to_dict()


def _extract_json(text: str) -> Dict[str, Any]:
    """Return parsed JSON dict from a model reply, or empty fallback."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return {}
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return {}

In [ ]:
# ── 6. Stub agents (always run — no API key needed) ──────────────────────────

def stub_noop(inst: Instance) -> Dict[str, Any]:
    claims = [
        {"rule_name": g["rule_name"], "component": g["component"],
         "known": True, "confidence": 0.5}
        for g in inst.metacog_ground_truth
    ]
    return {
        "metacog_assessment": claims,
        "critical_unknowns_ranked": [],
        "exploratory_actions": [],
        "final_plan": [{"kind": "wait"}],
        "self_judgment": {
            "robustness_score": 50,
            "risks_identified": [],
            "alternative_if_unknown_X": {},
        },
    }


def stub_random(inst: Instance) -> Dict[str, Any]:
    rng   = random.Random(inst.seed ^ 0xDEADBEEF)
    n     = inst.world.initial.n
    kinds = ["pulse", "damp", "shift", "unshift", "observe", "wait"]
    rand_act = lambda: {"kind": rng.choice(kinds), "i": rng.randrange(n)}
    return {
        "metacog_assessment": [
            {"rule_name": g["rule_name"], "component": g["component"],
             "known": rng.random() > 0.5, "confidence": rng.random()}
            for g in inst.metacog_ground_truth
        ],
        "critical_unknowns_ranked": list(inst.true_unknown_ranking[::-1]),
        "exploratory_actions": [rand_act() for _ in range(rng.randint(0, 2))],
        "final_plan":          [rand_act() for _ in range(rng.randint(1, 5))],
        "self_judgment": {
            "robustness_score": 40,
            "risks_identified": ["random baseline"],
            "alternative_if_unknown_X": {"unknown": "", "plan": [rand_act()]},
        },
    }


def stub_greedy(inst: Instance) -> Dict[str, Any]:
    """Oracle plan assuming no hidden rules — decent when hidden parts matter little."""
    _, best_plan = oracle_score(inst.world)
    return {
        "metacog_assessment": [
            {"rule_name": g["rule_name"], "component": g["component"],
             "known": True, "confidence": 0.9}
            for g in inst.metacog_ground_truth
        ],
        "critical_unknowns_ranked": [],
        "exploratory_actions": [],
        "final_plan": [{"kind": a.kind, "i": a.i, "j": a.j} for a in best_plan],
        "self_judgment": {
            "robustness_score": 70,
            "risks_identified": [],
            "alternative_if_unknown_X": {},
        },
    }

print("Stub agents defined.")

In [ ]:
# ── 7. Gemini agents (Google GenAI SDK) ──────────────────────────────────────
# Uses MODEL_PROXY_API_KEY (Kaggle-provisioned $50/day quota).

def _make_gemini_agent(model_id: str):
    """Factory: returns an agent function bound to *model_id*."""
    def agent(inst: Instance) -> Dict[str, Any]:
        import google.genai as genai
        client = genai.Client(api_key=MODEL_PROXY_API_KEY)
        prompt = build_prompt(inst)
        resp   = client.models.generate_content(
            model=model_id,
            contents=prompt,
            config=genai.types.GenerateContentConfig(
                max_output_tokens=2048,
                temperature=0.0,
            ),
        )
        return _extract_json(resp.text)
    agent.__name__ = model_id
    return agent


# All Gemini models available through the Kaggle proxy
GEMINI_MODELS = [
    "gemini-2.5-pro",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-1.5-pro",
    "gemini-1.5-flash",
    "gemini-1.5-flash-8b",
]

gemini_agents = {
    m: _make_gemini_agent(m) for m in GEMINI_MODELS
} if MODEL_PROXY_API_KEY else {}

print(f"Gemini agents ready: {list(gemini_agents.keys()) or 'NONE (no MODEL_PROXY_API_KEY)'}")

In [ ]:
# ── 8. Claude agents (Anthropic API) ─────────────────────────────────────────

def _make_claude_agent(model_id: str):
    def agent(inst: Instance) -> Dict[str, Any]:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        prompt = build_prompt(inst)
        msg    = client.messages.create(
            model=model_id,
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}],
        )
        text = "".join(
            b.text for b in msg.content
            if getattr(b, "type", "") == "text"
        )
        return _extract_json(text)
    agent.__name__ = model_id
    return agent


CLAUDE_MODELS = [
    "claude-opus-4-6",
    "claude-sonnet-4-6",
    "claude-haiku-4-5-20251001",
]

claude_agents = {
    m: _make_claude_agent(m) for m in CLAUDE_MODELS
} if ANTHROPIC_API_KEY else {}

print(f"Claude agents ready: {list(claude_agents.keys()) or 'NONE (no ANTHROPIC_API_KEY)'}")

In [ ]:
# ── 9. OpenAI agents (GPT-4o family) ─────────────────────────────────────────

def _make_openai_agent(model_id: str):
    def agent(inst: Instance) -> Dict[str, Any]:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        prompt = build_prompt(inst)
        resp   = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=2048,
            temperature=0.0,
        )
        return _extract_json(resp.choices[0].message.content or "")
    agent.__name__ = model_id
    return agent


OPENAI_MODELS = [
    "gpt-4o",
    "gpt-4o-mini",
    "o1-mini",
    "o3-mini",
]

openai_agents = {
    m: _make_openai_agent(m) for m in OPENAI_MODELS
} if OPENAI_API_KEY else {}

print(f"OpenAI agents ready: {list(openai_agents.keys()) or 'NONE (no OPENAI_API_KEY)'}")

In [ ]:
# ── 10. HuggingFace Inference Providers ──────────────────────────────────────

def _make_hf_agent(model_id: str, provider: str = "auto"):
    def agent(inst: Instance) -> Dict[str, Any]:
        from huggingface_hub import InferenceClient
        client = InferenceClient(model=model_id, token=HF_TOKEN, provider=provider)
        prompt = build_prompt(inst)
        resp   = client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=2048,
            temperature=0.0,
        )
        return _extract_json(resp.choices[0].message.content or "")
    agent.__name__ = model_id.split("/")[-1]
    return agent


HF_MODELS = [
    "meta-llama/Llama-3.3-70B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen2.5-72B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-70B",
]

hf_agents = {
    m.split("/")[-1]: _make_hf_agent(m) for m in HF_MODELS
} if HF_TOKEN else {}

print(f"HF agents ready: {list(hf_agents.keys()) or 'NONE (no HF_TOKEN)'}")

In [ ]:
# ── 11. Master agent registry ─────────────────────────────────────────────────

STUB_AGENTS = {
    "stub-noop":   stub_noop,
    "stub-random": stub_random,
    "stub-greedy": stub_greedy,
}

ALL_AGENTS = {
    **STUB_AGENTS,
    **gemini_agents,
    **claude_agents,
    **openai_agents,
    **hf_agents,
}

print(f"Total agents registered: {len(ALL_AGENTS)}")
for name in ALL_AGENTS:
    print(f"  {name}")

In [ ]:
# ── 12. Evaluation loop ───────────────────────────────────────────────────────

from tqdm.auto import tqdm

def evaluate_agent(name: str, agent_fn, records: List[Dict],
                   retry_on_error: bool = True) -> Dict[str, Any]:
    """
    Run *agent_fn* over *records*, score each instance, return summary dict.
    Skips instances where the agent raises an exception (logs them).
    """
    per_instance = []
    errors = 0

    for rec in tqdm(records, desc=f"{name:35s}", leave=False):
        inst = _instance_from_record(rec)
        try:
            raw  = agent_fn(inst)
            resp = validate_response(raw)
            best  = rec["hidden"].get("oracle_best")
            worst = rec["hidden"].get("oracle_worst")
            bd    = score_response(resp, inst, best_obj=best, worst_obj=worst)
            per_instance.append({
                "id":         inst.id,
                "difficulty": inst.difficulty,
                **bd.to_dict(),
            })
        except Exception as exc:
            errors += 1
            per_instance.append({
                "id":          inst.id,
                "difficulty":  inst.difficulty,
                "composite":   0.0,
                "objective":   0.0,
                "calibration": 0.0,
                "attention":   0.0,
                "executive":   0.0,
                "error":       str(exc),
            })

    n    = len(per_instance)
    avg  = lambda k: sum(r.get(k, 0) for r in per_instance) / max(n, 1)
    return {
        "agent":             name,
        "n":                 n,
        "errors":            errors,
        "mean_composite":    avg("composite"),
        "mean_objective":    avg("objective"),
        "mean_calibration":  avg("calibration"),
        "mean_attention":    avg("attention"),
        "mean_executive":    avg("executive"),
        "per_instance":      per_instance,
        "timestamp":         datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }


# Run all registered agents
RESULTS: Dict[str, Dict] = {}

for agent_name, agent_fn in ALL_AGENTS.items():
    print(f"\nEvaluating: {agent_name}")
    t0 = time.time()
    result = evaluate_agent(agent_name, agent_fn, EVAL_RECORDS)
    elapsed = time.time() - t0
    RESULTS[agent_name] = result
    print(
        f"  composite={result['mean_composite']:.3f}  "
        f"obj={result['mean_objective']:.3f}  "
        f"cal={result['mean_calibration']:.3f}  "
        f"att={result['mean_attention']:.3f}  "
        f"exe={result['mean_executive']:.3f}  "
        f"errors={result['errors']}  "
        f"({elapsed:.1f}s)"
    )

print("\nAll evaluations complete.")

In [ ]:
# ── 13. Leaderboard table ─────────────────────────────────────────────────────

import pandas as pd

rows = [
    {
        "Model":       r["agent"],
        "Composite ↑": round(r["mean_composite"],   3),
        "Objective":   round(r["mean_objective"],    3),
        "Calibration": round(r["mean_calibration"],  3),
        "Attention":   round(r["mean_attention"],    3),
        "Executive":   round(r["mean_executive"],    3),
        "N": r["n"],
        "Errors": r["errors"],
    }
    for r in sorted(RESULTS.values(), key=lambda x: -x["mean_composite"])
]

df_lb = pd.DataFrame(rows)
display(df_lb.to_string(index=False))

In [ ]:
# ── 14. Visualisation ─────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

DIMS   = ["Objective", "Calibration", "Attention", "Executive"]
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

models     = df_lb["Model"].tolist()
composites = df_lb["Composite ↑"].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(models) * 0.45)))

# Left: composite score bar chart
ax = axes[0]
y  = np.arange(len(models))
bars = ax.barh(y, composites, color="steelblue", edgecolor="white")
ax.set_yticks(y); ax.set_yticklabels(models, fontsize=9)
ax.set_xlabel("Composite Score (0–1)")
ax.set_title("CIPHER Leaderboard — Composite", fontweight="bold")
ax.set_xlim(0, 1.0)
for bar, val in zip(bars, composites):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=8)
ax.invert_yaxis()

# Right: radar / grouped bar of sub-scores
ax2    = axes[1]
x      = np.arange(len(DIMS))
width  = 0.8 / max(len(models), 1)
for idx, model in enumerate(models):
    row    = df_lb[df_lb["Model"] == model].iloc[0]
    scores = [row[d] for d in DIMS]
    offset = (idx - len(models)/2 + 0.5) * width
    ax2.bar(x + offset, scores, width,
            label=model, alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(DIMS)
ax2.set_ylim(0, 1.0)
ax2.set_ylabel("Score")
ax2.set_title("Sub-score Breakdown", fontweight="bold")
ax2.legend(fontsize=7, loc="upper right", ncol=max(1, len(models)//6))

plt.tight_layout()
plt.savefig("/kaggle/working/cipher_leaderboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to /kaggle/working/cipher_leaderboard.png")

In [ ]:
# ── 15. Save results JSON ─────────────────────────────────────────────────────

OUT_PATH = "/kaggle/working/cipher_all_results.json"
with open(OUT_PATH, "w") as f:
    json.dump(
        {
            "leaderboard": rows,
            "per_model":   {
                k: {"summary": {kk: vv for kk, vv in v.items() if kk != "per_instance"},
                    "per_instance": v["per_instance"]}
                for k, v in RESULTS.items()
            },
        },
        f, indent=2,
    )
print(f"Results written to {OUT_PATH}")

---
## Kaggle Benchmarks — Task Registration

The cells below register and run the official CIPHER Kaggle Benchmark tasks.
They **must** run inside a Kaggle Benchmark Task notebook to be counted.

One `@kbench.task` is defined per model family so the benchmark shows
per-model pass-rates directly on the Kaggle Benchmarks page.

After running, go to https://www.kaggle.com/benchmarks/tasks/new to finalize.

In [ ]:
# ── 16. kbench setup ──────────────────────────────────────────────────────────

import kaggle_benchmarks as kbench
from kaggle_benchmarks.actors.llms import GoogleGenAI
import google.genai as genai

# How many instances to register as official benchmark runs
# (use ALL 1 000 for the final submission; 50 for a quick test)
KBENCH_LIMIT = EVAL_LIMIT   # change to len(ALL_RECORDS) for full submission

bench_records = ALL_RECORDS[:KBENCH_LIMIT]
print(f"Will register {len(bench_records)} kbench runs per model.")

In [ ]:
# ── 17. Define the CIPHER kbench task ────────────────────────────────────────
#
# Each .run() call = one instance evaluated on one LLM.
# The task passes/fails based on composite >= 0.5.
# Kaggle Benchmarks records the pass-rate across all runs.

@kbench.task(name="cipher_metacog")
def cipher_task(
    llm,
    instance_id: str,
    prompt: str,
    record_json: str,
):
    """
    CIPHER metacognition task.

    The LLM receives a partially-hidden causal world described in invented
    vocabulary and must return strict JSON with:
      - metacog_assessment   (calibrated self-knowledge)
      - critical_unknowns_ranked  (attention to high-impact unknowns)
      - exploratory_actions  (probe budget)
      - final_plan           (committed action sequence)
      - self_judgment        (executive function: risks, alternatives)

    Scored on four axes: Objective | Calibration | Attention | Executive.
    Passes when composite score >= 0.5.
    """
    response  = llm.prompt(prompt)
    rec       = json.loads(record_json)
    breakdown = score_text_response(str(response), rec)
    composite = breakdown.get("composite", 0.0)

    kbench.assertions.assert_true(
        composite >= 0.5,
        f"Instance {instance_id}: composite {composite:.3f} < 0.5. "
        f"obj={breakdown.get('objective',0):.3f}  "
        f"cal={breakdown.get('calibration',0):.3f}  "
        f"att={breakdown.get('attention',0):.3f}  "
        f"exe={breakdown.get('executive',0):.3f}",
    )

print("cipher_task defined.")

In [ ]:
# ── 18. Run kbench tasks for every Gemini model ───────────────────────────────
# Kaggle's MODEL_PROXY_API_KEY gives access to all Gemini tiers for free.

if not MODEL_PROXY_API_KEY:
    print("Skipping kbench runs — MODEL_PROXY_API_KEY not set.")
else:
    _genai_client = genai.Client(api_key=MODEL_PROXY_API_KEY)

    for model_id in GEMINI_MODELS:
        print(f"\n── kbench: {model_id} ({len(bench_records)} instances) ──")
        llm = GoogleGenAI(
            client=_genai_client,
            model=model_id,
        )
        for rec in tqdm(bench_records, desc=model_id, leave=False):
            try:
                cipher_task.run(
                    llm=llm,
                    instance_id=rec["id"],
                    prompt=rec["prompt"],
                    record_json=json.dumps(rec),
                )
            except Exception as exc:
                print(f"  WARN {rec['id']}: {exc}")

        print(f"  Done: {model_id}")

    print("\nAll Gemini kbench runs complete.")

In [ ]:
# ── 19. (Optional) Run kbench tasks for stub baselines ────────────────────────
#
# Stubs don't use the kbench LLM interface — we monkeypatch a fake LLM.
# This lets the Kaggle Benchmark page show stub scores for reference.

class _StubLLM:
    """Wraps a stub agent so it looks like a kbench LLM actor."""
    def __init__(self, name: str, fn):
        self.name = name
        self._fn  = fn
    def prompt(self, prompt_text: str) -> str:
        # We can't reconstruct the Instance from the prompt text alone,
        # so stubs are evaluated differently in cell 12.  Here we return
        # a valid but minimal JSON so kbench can parse a response.
        return json.dumps({
            "metacog_assessment": [],
            "critical_unknowns_ranked": [],
            "exploratory_actions": [],
            "final_plan": [{"kind": "wait"}],
            "self_judgment": {
                "robustness_score": 50,
                "risks_identified": [],
                "alternative_if_unknown_X": {},
            },
        })


for stub_name in ["stub-noop", "stub-random", "stub-greedy"]:
    print(f"\n── kbench stub: {stub_name} ──")
    stub_llm = _StubLLM(stub_name, ALL_AGENTS[stub_name])
    for rec in tqdm(bench_records, desc=stub_name, leave=False):
        try:
            cipher_task.run(
                llm=stub_llm,
                instance_id=rec["id"],
                prompt=rec["prompt"],
                record_json=json.dumps(rec),
            )
        except Exception as exc:
            pass  # stubs may score below 0.5; that's expected

print("\nStub kbench runs complete.")

In [ ]:
# ── 20. Final summary ─────────────────────────────────────────────────────────

print("\n" + "="*70)
print("CIPHER — FINAL LEADERBOARD")
print("="*70)
print(df_lb.to_string(index=False))
print("\nNext steps:")
print("  1. Run all cells with KBENCH_LIMIT = len(ALL_RECORDS) for full 1000-instance eval.")
print("  2. Go to https://www.kaggle.com/benchmarks/tasks/new and publish the 'cipher_metacog' task.")
print("  3. Create a Benchmark collection named 'CIPHER' and add the task.")
print("  4. Attach the benchmark URL to your competition writeup.")